In [1]:
import torch
import torch.distributions as distr
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import plotly.express as px
import pandas as pd
from copy import deepcopy

/Users/ryanpegoud/Documents/Projects/ember/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
d1 = distr.Normal(-2, 0.5)
d2 = distr.Normal(2, 0.5)


def sample_target(batch_size: int) -> torch.Tensor:
    assert batch_size % 2 == 0, f"Batch size must be even, got {batch_size}"

    return torch.cat(
        [
            d1.sample((batch_size // 2, 1)),
            d2.sample((batch_size // 2, 1)),
        ]
    )


def sample_noise(batch_size: int) -> torch.Tensor:
    return distr.Normal(0, 1).sample((batch_size, 1))


batch_size = 16

In [3]:
samples = 8192 * 2
targets = pd.DataFrame(sample_target(samples).flatten().numpy())
noise = pd.DataFrame(sample_noise(samples).flatten().numpy())

targets["type"] = "target"
noise["type"] = "noise"
df = pd.concat((noise, targets), axis=0)
df.rename(columns={0: "value"}, inplace=True)

px.histogram(
    df,
    x="value",
    color="type",
    facet_col="type",
    template="plotly_white",
    title="Initial distributions",
)

## Probability Path
Here we decide how the points should move. For this toy example, they follow a **linear probability path**, we interpolate a noise sample $ x_{0} $ and a target sample $ x_{1} $:  

$$
    x_{t} = (1-t)x_{0} + tx_{1}
$$

## Vector Field
Our goal is to learn a **time-dependent vector field** $ v_{t}(x) $ describing how samples flow between the noise distribution (t=0) to the target distribution (t=1). This is governed by an ODE:
$$
    \frac{dx}{dt} = v_{t}(x)
$$
In order to sample from the target distribution, we define a **conditional vector field** $ u_{t}(x| x_{1}) $, which we approximate with a neural network $ v_{\theta}(x,t) $: 
$$
  u_{t}(x| x_{1}) =  \frac{dx_{t}}{dt} = x_{1} - x_{0}
$$ 

In [4]:
def get_noisy_state_and_velocity(
    batch_size: int, t: torch.Tensor | None = None
) -> torch.Tensor:
    assert batch_size % 2 == 0, f"Batch size must be even, got {batch_size}"
    if not t:
        t = distr.Uniform(0, 1).sample((batch_size, 1))
    x0 = sample_noise(batch_size)
    x1 = sample_target(batch_size)

    noisy_state = torch.cat((t, (1 - t) * x0 + t * x1), axis=1)
    velocity = x1 - x0
    return noisy_state, velocity

In [5]:
hidden_size = 256
train_samples = 8192

model = nn.Sequential(
    nn.Linear(2, hidden_size),
    nn.SiLU(),
    nn.Linear(hidden_size, hidden_size),
    nn.SiLU(),
    nn.Linear(hidden_size, 1),
)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

losses = []
for i in tqdm(range(train_samples // batch_size)):
    optimizer.zero_grad()
    noisy_states, velocity = get_noisy_state_and_velocity(batch_size)
    preds = model(noisy_states)
    loss = F.mse_loss(preds, velocity)
    losses.append(loss.item())
    loss.backward()
    optimizer.step()

px.line(losses, title="MSE Loss", template="plotly_white")

100%|██████████| 512/512 [00:00<00:00, 1578.92it/s]


## Sampling
To sample from the learned target distribution, we define a noise schedule and update the noisy sample iteratively using the Euler method.
Here, the update is given by:
$$
    x_{t + \Delta t} = x_t + v_\theta(x_t, t) \cdot \Delta t
$$

In [6]:
tensor_to_numpy = lambda x: x.detach().flatten().numpy()

@torch.no_grad
def sample(batch_size: int, n_steps: int) -> torch.Tensor:
    noise_schedule = torch.linspace(0, 1, n_steps)
    delta_t = 1 / n_steps

    x_0 = sample_noise(batch_size)
    x_t = deepcopy(x_0)

    for idx, t in tqdm(enumerate(noise_schedule)):
        if idx == n_steps -1:  # skip the last step
            continue
        t = t.broadcast_to(x_t.shape)
        predicted_velocity = model(torch.cat((t, x_t), dim=1))
        x_t += predicted_velocity * delta_t

    return x_0, x_t


x_0, x_t = sample(batch_size=2000, n_steps=50)

x_0, x_t = tuple(map(tensor_to_numpy, (x_0, x_t)))
noise = pd.DataFrame(x_0)
targets = pd.DataFrame(x_t)

targets["type"] = "target"
noise["type"] = "noise"
df = pd.concat((noise, targets), axis=0)
df.rename(columns={0: "value"}, inplace=True)

px.histogram(
    df,
    x="value",
    color="type",
    facet_col="type",
    title="Flow-matching samples",
    template="plotly_white",
)

50it [00:00, 1012.95it/s]
